## Задача


\begin{cases}
\dfrac{dx}{dt} = x \left( 1 - 0.5x - \dfrac{2}{7} \alpha_2^{-2} y \right), \quad x(0) = x_0, \\[10pt]
\dfrac{dy}{dt} = y \left( 2\alpha_2 - 0.5y - 3.5\alpha_2^2 x \right), \quad y(0) = y_0, \\[10pt]
\dfrac{d\alpha_2}{dt} = \varepsilon (2 - 7\alpha_2 x), \quad \alpha_2(0) = \alpha_{20}; \quad t \in [0; T_k].
\end{cases}

#### Рекомендуемые значения начальных данных


$0 \le x_0 \le 3$,

 $0 \le y_0 \le 15$
 
 $\alpha_{20}$ близко к нулю, например,
можно положить $\alpha_{20} = 0.0001$;

 $T_k = 1500$. 
 
$\varepsilon \le 0.01$ 

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d
from numba import njit
import time
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display


from warnings import filterwarnings
filterwarnings("ignore", category=UserWarning)

In [2]:
@njit
def system_func(t, v, epsilon=0.01):
    x, y, a2 = v
    
    # Если популяция ушла в минус - она мертва (0)
    if x < 1e-10: x = 0.0
    if y < 1e-10: y = 0.0
        
    a2_reg = a2 if abs(a2) > 1e-5 else 1e-5 

    dxdt = x * (1 - 0.5 * x - (2/7) * (a2_reg**-2) * y)
    dydt = y * (2 * a2 - 0.5 * y - 3.5 * (a2_reg**2) * x)
    da2dt = epsilon * (2 - 7 * a2 * x)
    
    # Мертвые не размножаются
    if x == 0.0: dxdt = 0.0
    if y == 0.0: dydt = 0.0
        
    return np.array([dxdt, dydt, da2dt])

In [3]:
@njit
def rk4_step_fast(f, t, y, h):
    k1 = f(t, y)
    k2 = f(t + h/2, y + h/2 * k1)
    k3 = f(t + h/2, y + h/2 * k2)
    k4 = f(t + h, y + h * k3)
    return y + h/6 * (k1 + 2*k2 + 2*k3 + k4)

@njit
def adams_b(f, y0, t_span, h):
    n_steps = int(np.ceil((t_span[1] - t_span[0]) / h)) + 1
    t = np.linspace(t_span[0], t_span[1], n_steps)
    y = np.zeros((n_steps, len(y0)))
    y[0] = y0
    
    for i in range(3):
        y[i+1] = rk4_step_fast(f, t[i], y[i], h)
        
    f0 = f(t[0], y[0])
    f1 = f(t[1], y[1])
    f2 = f(t[2], y[2])
    f3 = f(t[3], y[3])
    
    for i in range(3, n_steps - 1):
        y[i+1] = y[i] + (h/24) * (55*f3 - 59*f2 + 37*f1 - 9*f0)
        f0, f1, f2, f3 = f1, f2, f3, f(t[i+1], y[i+1])
        
    return t, y

@njit
def adams_m(f, y0, t_span, h, tol=1e-7, max_iter=100):
    n_steps = int(np.ceil((t_span[1] - t_span[0]) / h)) + 1
    t = np.linspace(t_span[0], t_span[1], n_steps)
    y = np.zeros((n_steps, len(y0)))
    y[0] = y0
    
    for i in range(3):
        y[i+1] = rk4_step_fast(f, t[i], y[i], h)
    f1 = f(t[1], y[1])
    f2 = f(t[2], y[2])
    f3 = f(t[3], y[3])
    
    cnt = 0
    for i in range(3, n_steps - 1):
        y_guess = y[i] + h * f3
        
        for _ in range(max_iter):
            cnt+=1
            y_new = y[i] + (h/24) * (9*f(t[i+1], y_guess) + 19*f3 - 5*f2 + f1)
            
            if np.linalg.norm(y_new - y_guess) < tol:
                break
            y_guess = y_new
            
        y[i+1] = y_new
        f1, f2, f3 = f2, f3, f(t[i+1], y[i+1])
    print("ADAMS_M: In average, it took", cnt//(n_steps-4), "iterations per step for convergence.")
    return t, y

@njit
def adams_bm(f, y0, t_span, h):
    n_steps = int(np.ceil((t_span[1] - t_span[0]) / h)) + 1
    t = np.linspace(t_span[0], t_span[1], n_steps)
    y = np.zeros((n_steps, len(y0)))
    y[0] = y0
    
    for i in range(3):
        y[i+1] = rk4_step_fast(f, t[i], y[i], h)
        
    f0 = f(t[0], y[0])
    f1 = f(t[1], y[1])
    f2 = f(t[2], y[2])
    f3 = f(t[3], y[3])
    
    for i in range(3, n_steps - 1):
        y_predict = y[i] + (h/24) * (55*f3 - 59*f2 + 37*f1 - 9*f0)
        f_predict = f(t[i+1], y_predict)
        y[i+1] = y[i] + (h/24) * (9*f_predict + 19*f3 - 5*f2 + f1)
        f0, f1, f2, f3 = f1, f2, f3, f(t[i+1], y[i+1])
    return t, y

## Исследование на начальных данных

In [15]:

def plot_interactive_system(x0, y0, a20, epsilon=0.01):
    t_span = [0.0, 2000.0]
    h = 0.00001
    y0_arr = np.array([x0, y0, a20], dtype=np.float64)
    
    system_func_with_epsilon = lambda t, v: system_func(t, v, epsilon)
    
    # t_ab, y_ab = adams_b(system_func_with_epsilon, y0_arr, t_span, h)
    # t_am, y_am = adams_m(system_func_with_epsilon, y0_arr, t_span, h)
    # t_abm, y_abm = adams_bm(system_func_with_epsilon, y0_arr, t_span, h)
    t_s, y_s = solve_ivp(system_func_with_epsilon, t_span, y0_arr, method='BDF', atol=1e-12, rtol=1e-12).t, solve_ivp(system_func_with_epsilon, t_span, y0_arr, method='BDF', atol=1e-12, rtol=1e-12).y.T
    
    
    fig = go.Figure()

    def add_traces(t, y, method_name, line_dash):
        if np.any(np.isnan(y)) or np.any(np.isinf(y)) or np.any(y < 0) or np.any(y > 1e3):
            print(f"Внимание: Метод {method_name} ВЗОРВАЛСЯ при этих данных!")
            return

        N = 50 
        t_plot = t[::N]
        y_plot = y[::N]
        
        # Рисуем уже прореженные массивы t_plot и y_plot
        fig.add_trace(go.Scatter(
            x=t_plot, y=y_plot[:, 0], mode='lines', 
            name=f'Вид X ({method_name})',
            line=dict(color='red', width=2, dash=line_dash) 
        ))
        fig.add_trace(go.Scatter(
            x=t_plot, y=y_plot[:, 1], mode='lines', 
            name=f'Вид Y ({method_name})',
            line=dict(color='blue', width=2, dash=line_dash) 
        ))
        
    # add_traces(t_ab, y_ab, "AB4", "solid")
    # add_traces(t_am, y_am, "AM4", "dash")
    # add_traces(t_abm, y_abm, "ABM", "dot")
    add_traces(t_s, y_s, "BDF", "dashdot")

    fig.update_layout(
        title=f"x0={x0:.1f}, y0={y0:.1f}, α20={a20:.3f}",
        xaxis_title="Время (t)",
        yaxis_title="Численность",
        template="plotly_white",
        height=600,
        hovermode="x unified",
        yaxis=dict(range=[0, max(x0, y0) * 1.5 + 5]) 
    )
    
    fig.show()


widgets.interact(
    plot_interactive_system,
    x0=widgets.FloatSlider(value=1.5, min=0.0, max=3.0, step=0.1, description='Вид X (x0):',continuous_update=False),
    y0=widgets.FloatSlider(value=7.5, min=0.0, max=15.0, step=0.5, description='Вид Y (y0):',continuous_update=False),
    a20=widgets.FloatSlider(value=0.001, min=0.00001, max=2, step=0.01, description='Ген α20:', readout_format='.4f', continuous_update=False),
    epsilon=widgets.FloatSlider(value=0.01, min=0.001, max=1, step=0.001, description='Epsilon:', readout_format='.3f', continuous_update=False)
    
)

interactive(children=(FloatSlider(value=1.5, continuous_update=False, description='Вид X (x0):', max=3.0), Flo…

<function __main__.plot_interactive_system(x0, y0, a20, epsilon=0.01)>

In [5]:

def plot_interactive_system(x0, y0, a20):
    t_span = [0.0, 100.0]
    h = 0.0005
    y0_arr = np.array([x0, y0, a20], dtype=np.float64)
    
    t_ab, y_ab = adams_b(system_func, y0_arr, t_span, h)
    t_am, y_am = adams_m(system_func, y0_arr, t_span, h)
    t_abm, y_abm = adams_bm(system_func, y0_arr, t_span, h)

    
    
    
    fig = go.Figure()

    def add_traces(t, y, method_name, line_dash):
        if np.any(np.isnan(y)) or np.any(np.isinf(y)):
            print(f"Внимание: Метод {method_name} ВЗОРВАЛСЯ при этих данных!")
            return

        N = 50 
        t_plot = t[::N]
        y_plot = y[::N]
        
        # Рисуем уже прореженные массивы t_plot и y_plot
        fig.add_trace(go.Scatter(
            x=t_plot, y=y_plot[:, 0], mode='lines', 
            name=f'Вид X ({method_name})',
            line=dict(color='red', width=2, dash=line_dash) 
        ))
        fig.add_trace(go.Scatter(
            x=t_plot, y=y_plot[:, 1], mode='lines', 
            name=f'Вид Y ({method_name})',
            line=dict(color='blue', width=2, dash=line_dash) 
        ))
        
    add_traces(t_ab, y_ab, "AB4", "solid")
    add_traces(t_am, y_am, "AM4", "dash")
    add_traces(t_abm, y_abm, "ABM", "dot")

    fig.update_layout(
        title=f"x0={x0:.1f}, y0={y0:.1f}, α20={a20:.3f}",
        xaxis_title="Время (t)",
        yaxis_title="Численность",
        template="plotly_white",
        height=600,
        hovermode="x unified",
        yaxis=dict(range=[0, max(x0, y0) * 1.5 + 5]) 
    )
    
    fig.show()


widgets.interact(
    plot_interactive_system,
    x0=widgets.FloatSlider(value=1.5, min=0.1, max=5.0, step=0.1, description='Вид X (x0):'),
    y0=widgets.FloatSlider(value=7.5, min=0.1, max=20.0, step=0.5, description='Вид Y (y0):'),
    a20=widgets.FloatSlider(value=0.01, min=0.00001, max=1.0, step=0.001, description='Ген α20:', readout_format='.3f')
)

interactive(children=(FloatSlider(value=1.5, description='Вид X (x0):', max=5.0, min=0.1), FloatSlider(value=7…

<function __main__.plot_interactive_system(x0, y0, a20)>